In [2]:
import re
import sys
import os
import cv2
import numpy as np

from marsneuralzoo.models.e2emvm import E2emvm, GraphTracker
from marsneuralzoo.models.yolo_seg import SegmentationYolo

In [3]:
def read_data(data_path):
    folders = os.listdir(data_path)

    folders = [f for f in folders if os.path.isdir(os.path.join(data_path, f))]
    folders.sort(key=lambda f: f)
    folders = [os.path.join(data_path, f) for f in folders]

    image_path_list = []
    segments_path_list = []
    confidences_path_list = []

    for folder in folders:
        files = os.listdir(folder)

        image_names = [f for f in files if f.endswith(".jpeg")]
        image_names.sort(key=lambda p: int(os.path.splitext(p)[0]))
        image_paths = [os.path.join(folder, f) for f in image_names]

        image_path_list.append(image_paths)

        mask_names = [f for f in files if f.endswith("_mask.npy")]
        mask_names.sort(key=lambda p: int(re.search(r'\d+', p).group()))
        mask_paths = [os.path.join(folder, f) for f in mask_names]

        confidences_names = [f for f in files if f.endswith("_confs.npy")]
        confidences_names.sort(key=lambda p: int(re.search(r'\d+', p).group()))
        confidences_paths = [os.path.join(folder, f) for f in confidences_names]
        # segments_paths_s[folder] = mask_paths
        segments_path_list.append(mask_paths)
        confidences_path_list.append(confidences_paths)

    image_paths_list = [list(pair) for pair in zip(*image_path_list)]
    segments_paths_list = [list(pair) for pair in zip(*segments_path_list)]
    confidences_paths_list = [
        list(pair) for pair in zip(*confidences_path_list)
    ]

    return image_paths_list, segments_paths_list, confidences_paths_list

In [4]:
e2emvm = E2emvm(multiview=False)
graph_tracker = GraphTracker(track_stride=3)
obj_seg_model = SegmentationYolo()

Loaded SuperPoint model


In [13]:
data_path = '/home/mars/Desktop/temp/2023_04_18_09_29_56_seg_0'
data_path = '/home/mars/Desktop/temp/2023_04_27_17_07_32_seg_13'
data_base_name = os.path.basename(data_path)

os.makedirs(f'./cache/{data_base_name}', exist_ok=True)

image_paths_list, segments_paths_list, confidences_paths_list = read_data(
    data_path
)

In [6]:
from dataengine.generator.obstacle.debug_utils import (
    draw_mask,
    visualize_optical_flow,
)
from dataengine.generator.obstacle.assignment import assign_segments

from dataengine.generator.obstacle.segment_tracker import (
    CamId,
    SegmentTracklet,
    SegmentTracker,
    TrackletDatabase,
)

In [7]:
from typing import Dict, List


class CamState:
    def __init__(self):
        self.cam_id: CamId = CamId()
        self.image: np.ndarray = None  # H x W x 3
        self.gray: np.ndarray = None  # H x W

        self.segs: np.ndarray = None  # n x H x W
        self.confs: np.ndarray = None  # n x 1
        self.seg_trls: Dict[int, SegmentTracklet] = {}

        self.features = None  # di  ctionary from sperpoint

    def draw_id(self):
        shape = self.image.shape[:2]
        id_image = np.full(shape, -1, dtype=int)

        for id, seg_trl in self.seg_trls.items():
            mask = seg_trl.query_segment_mask(self.cam_id)
            id_image[mask > 0] = seg_trl.id
        return id_image

    def draw_merged_tracklet(self):

        # input_segment_image = cv2.cvtColor(self.image, cv2.COLOR_RGB2BGR).copy()
        image = cv2.cvtColor(self.image, cv2.COLOR_RGB2BGR).copy()

        for seg_id, seg_trl in self.seg_trls.items():
            if not seg_trl.merged:
                continue
            mask = seg_trl.query_segment_mask(self.cam_id)
            image = draw_mask(image, mask, seg_id, 0.8)

        # seg_idx = 0
        # for seg in self.segs:
        #     input_segment_image = draw_mask(
        #         input_segment_image, seg, seg_idx, 0.8
        #     )
        #     seg_idx += 1

        description = f"{self.cam_id.timestamp} - {self.cam_id.index}"
        cv2.putText(
            image,
            description,
            (40, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0, 255, 0),
            1,
            cv2.LINE_AA,
        )
        out_image = image
        # out_image = np.concatenate((input_segment_image, image), axis=1)
        # W = out_image.shape[1] // 2
        # out_image[:, W - 1 : W, :] = np.array([255, 255, 255], dtype=np.uint8)

        return out_image

    def draw_tracklet(self):
        """
        Draw tracked segments for debug. Left side is the input image and segments and
        right side shows tracked segments with unique color based on tracklet id
        """
        image = cv2.cvtColor(self.image, cv2.COLOR_RGB2BGR).copy()

        for seg_id, seg_trl in self.seg_trls.items():
            if seg_trl.track_count() < 2:
                continue
            mask = seg_trl.query_segment_mask(self.cam_id)
            image = draw_mask(image, mask, seg_id, 0.8)

        description = f"{self.cam_id.timestamp} - {self.cam_id.index}"
        cv2.putText(
            image,
            description,
            (40, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0, 255, 0),
            1,
            cv2.LINE_AA,
        )
        out_image = image

        return out_image

    def draw_scene(self):
        """
        Draw tracked segments for debug. Left side is the input image and segments and
        right side shows tracked segments with unique color based on tracklet id
        """
        input_segment_image = cv2.cvtColor(self.image, cv2.COLOR_RGB2BGR).copy()
        image = cv2.cvtColor(self.image, cv2.COLOR_RGB2BGR).copy()

        for seg_id, seg_trl in self.seg_trls.items():
            # if seg_trl.track_count() < 2:
            # continue
            mask = seg_trl.query_segment_mask(self.cam_id)
            image = draw_mask(image, mask, seg_trl.id, 0.8)

        seg_idx = 0
        for seg in self.segs:
            input_segment_image = draw_mask(
                input_segment_image, seg, seg_idx, 0.8
            )
            seg_idx += 1

        description = f"{self.cam_id.timestamp} - {self.cam_id.index}"
        cv2.putText(
            image,
            description,
            (40, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0, 255, 0),
            1,
            cv2.LINE_AA,
        )

        out_image = np.concatenate((input_segment_image, image), axis=1)
        W = out_image.shape[1] // 2
        out_image[:, W - 1 : W, :] = np.array([255, 255, 255], dtype=np.uint8)

        return out_image

In [8]:
video_layout = [
    [1664, 0],
    [0, 0],
    [832, 0],
    [0, 936],
    [1664, 936],
    [0, 468],
    [1664, 468],
]

IMG_W = int(832 * 3)
IMG_H = int(468 * 3)
img_w = int(832)
img_h = int(468)

In [9]:
K_list = []

k = np.array([[838.723, 0.0, 415.5], [0.0, 838.723, 233.5], [0.0, 0.0, 1.0]])
K_list.append(k)
k = np.array([[1701.793, 0.0, 415.5], [0.0, 1701.793, 233.5], [0.0, 0.0, 1.0]])
K_list.append(k)
k = np.array([[220.465, 0.0, 415.5], [0.0, 220.465, 233.5], [0.0, 0.0, 1.0]])
K_list.append(k)

k = np.array([[448.461, 0.0, 415.5], [0.0, 448.461, 233.5], [0.0, 0.0, 1.0]])
K_list.append(k)
k = np.array([[442.053, 0.0, 415.5], [0.0, 442.053, 233.5], [0.0, 0.0, 1.0]])
K_list.append(k)
k = np.array([[863.395, 0.0, 415.5], [0.0, 863.395, 233.5], [0.0, 0.0, 1.0]])
K_list.append(k)
k = np.array([[850.053, 0.0, 415.5], [0.0, 850.053, 233.5], [0.0, 0.0, 1.0]])
K_list.append(k)

distort_coef_list = []
d = np.array([-0.229, 0.039, -0.155, 0.0])
distort_coef_list.append(d)
d = np.array([-0.065, -0.104, 1.963, 0.0])
distort_coef_list.append(d)
d = np.array([0.134, -0.033, 0.0, 0.0])
distort_coef_list.append(d)

d = np.array([-0.085, -0.02, 0.005, 0.0])
distort_coef_list.append(d)
d = np.array([-0.05, -0.02, 0.005, 0.0])
distort_coef_list.append(d)
d = np.array([-0.21, -0.115, 0.205, 0.0])
distort_coef_list.append(d)
d = np.array([-0.19, 0.0, 0.005, 0.0])
distort_coef_list.append(d)

T_bc_list = []
T_bc = np.array(
    [
        [-0.02, -0.05, 0.999, 0.1],
        [1.0, -0.001, 0.02, 0.068],
        [0.0, 0.999, 0.051, -0.0],
        [0.0, 0.0, 0.0, 1.0],
    ]
)
T_bc_list.append(T_bc)
T_bc = np.array(
    [
        [-0.011, -0.036, 0.999, 0.104],
        [1.0, -0.001, 0.011, -0.061],
        [0.0, 0.999, 0.036, 0.0],
        [0.0, 0.0, 0.0, 1.0],
    ]
)
T_bc_list.append(T_bc)
T_bc = np.array(
    [
        [-0.045, 0.015, 0.999, 0.107],
        [0.999, 0.002, 0.045, 0.004],
        [-0.002, 1.0, -0.015, -0.001],
        [0.0, 0.0, 0.0, 1.0],
    ]
)
T_bc_list.append(T_bc)

T_bc = np.array(
    [
        [0.946, -0.137, 0.296, -1.673],
        [0.325, 0.44, -0.837, -1.196],
        [-0.016, 0.887, 0.461, 0.389],
        [0.0, 0.0, 0.0, 1.0],
    ]
)
T_bc_list.append(T_bc)
T_bc = np.array(
    [
        [-0.953, -0.122, 0.278, -1.658],
        [0.304, -0.375, 0.876, 1.204],
        [-0.003, 0.919, 0.394, 0.389],
        [0.0, 0.0, 0.0, 1.0],
    ]
)
T_bc_list.append(T_bc)
T_bc = np.array(
    [
        [0.451, 0.126, -0.884, -0.373],
        [-0.892, 0.032, -0.451, -1.296],
        [-0.028, 0.992, 0.127, 0.489],
        [0.0, 0.0, 0.0, 1.0],
    ]
)
T_bc_list.append(T_bc)
T_bc = np.array(
    [
        [-0.462, 0.127, -0.878, -0.368],
        [-0.886, -0.016, 0.464, 1.304],
        [0.045, 0.992, 0.119, 0.489],
        [0.0, 0.0, 0.0, 1.0],
    ]
)
T_bc_list.append(T_bc)


def skew(vector):
    x, y, z = vector
    return np.array([[0, -z, y], [z, 0, -x], [-y, x, 0]])

In [10]:
from marsdataio.npyrenderer.renderer import generate_colors

colors = generate_colors()

In [21]:
import torch
from itertools import count
from lapsolver import solve_dense

if 'torch_images' in globals():
    del torch_images, features, preds
    torch.cuda.empty_cache()


SegmentTracklet.id_counter = count()
tracklet_database = TrackletDatabase()
segment_trackers = [
    SegmentTracker(tracklet_database) for _ in range(len(image_paths_list[0]))
]

cam_states_dict = {}
segment_id_pairs_dict = {}

skip = 0
dura = 0
timestamp_count = 0

for image_paths in image_paths_list:
    timestamp_count += 1
    if timestamp_count < skip:
        continue

    if skip + dura != 0 and timestamp_count > skip + dura:
        break

    file_name = os.path.basename(image_paths[0])
    timestamp = int(os.path.splitext(file_name)[0])

    image_list = [cv2.imread(image_path) for image_path in image_paths]
    seg_results_list = obj_seg_model.segment_batch(image_list)

    cam_states = []
    print(f"processing {timestamp}")
    for cam_idx in range(len(image_list)):
        # if cam_idx != 3:
        # continue
        cam_state = CamState()
        cam_state.cam_id.timestamp = timestamp
        cam_state.cam_id.index = cam_idx
        cam_state.image = image_list[cam_idx]
        cam_state.segs = seg_results_list[cam_idx]['masks']

        confidences = np.array(seg_results_list[cam_idx]['scores'])
        confidences = confidences.reshape(-1, 1)
        cam_state.confs = confidences

        obstacle_tracker = segment_trackers[cam_idx]
        obstacle_tracker.track(cam_state)

        cam_states.append(cam_state)

    cam_states_dict[timestamp] = cam_states

    torch_images, features = e2emvm.extract_features(image_list)

    target_pair = [
        # front
        (0, 1),
        (0, 2),
        # (1, 2),
        # left
        (2, 3),
        (3, 5),
        # right
        (2, 4),
        (4, 6),
    ]

    preds = e2emvm(torch_images, features, target_pair)

    all_matches = preds['matches']

    kpt_seg_ids = []
    undistorted_kpts_list = []
    for cam_idx, kpts in enumerate(features['keypoints']):
        kpts = kpts.cpu().numpy().astype(np.float32)
        seg_ids = np.full(kpts.shape[0], -1, dtype=int)
        id_image = cam_states_dict[timestamp][cam_idx].draw_id()

        for idx, kpt in enumerate(kpts):
            x, y = kpt.astype(int)
            seg_id = id_image[y, x]
            if seg_id >= 0:
                seg_ids[idx] = seg_id

        kpt_seg_ids.append(seg_ids)

        K = K_list[cam_idx]
        D = distort_coef_list[cam_idx]
        R = np.eye(3, dtype=np.float32)
        P = np.eye(3, dtype=np.float32)
        # print(kpts.dtype)
        kpts = kpts.reshape(-1, 1, 2)

        undistorted_kpts = cv2.fisheye.undistortPoints(kpts, K, D, R=R, P=P)
        undistorted_kpts = undistorted_kpts.reshape(-1, 2)
        undistorted_kpts = np.hstack(
            (undistorted_kpts, np.ones((undistorted_kpts.shape[0], 1)))
        )
        undistorted_kpts_list.append(undistorted_kpts)

    segment_id_pair = []
    for idx0, idx1 in preds['indices_pairs']:
        key = f'{idx0}_{idx1}'

        matches = all_matches[f'matches{idx0}_{key}'][0].cpu().numpy()
        confs = all_matches[f'conf_scores_{key}'][0, :, 0].cpu().numpy()

        seg_ids0 = kpt_seg_ids[idx0]
        seg_ids1 = kpt_seg_ids[idx1]

        # Extract kpts0 based on match pairs, confidence, and whether they are within segment boundaries
        valid_indices = np.flatnonzero(
            (matches >= 0) & (confs >= 0.02) & (seg_ids0 >= 0)
        )

        kpts0 = features['keypoints'][idx0].cpu().numpy()
        kpts1 = features['keypoints'][idx1].cpu().numpy()

        kpts0 = kpts0[valid_indices]
        seg_ids0 = seg_ids0[valid_indices]
        undists0 = undistorted_kpts_list[idx0][valid_indices]

        kpts1_idx = matches[valid_indices]
        kpts1 = kpts1[kpts1_idx]
        seg_ids1 = seg_ids1[kpts1_idx]
        undists1 = undistorted_kpts_list[idx1][kpts1_idx]

        # Extract kpts1 based on whether they are within segment boundaries
        valid_indices = np.flatnonzero(seg_ids1 >= 0)

        kpts0 = kpts0[valid_indices]
        seg_ids0 = seg_ids0[valid_indices]
        undists0 = undists0[valid_indices]

        kpts1 = kpts1[valid_indices]
        seg_ids1 = seg_ids1[valid_indices]
        undists1 = undists1[valid_indices]

        T_bc0 = T_bc_list[idx0]
        T_bc1 = T_bc_list[idx1]
        R_bc0 = T_bc0[:3, :3]
        P_bc0 = T_bc0[:3, 3]
        R_bc1 = T_bc1[:3, :3]
        P_bc1 = T_bc1[:3, 3]

        R_c1b = R_bc1.T
        P_c1b = -R_c1b @ P_bc1

        R_c1c0 = R_c1b @ R_bc0
        P_c1c0 = R_c1b @ P_bc0 + P_c1b
        P_skew = skew(P_c1c0)
        E = P_skew @ R_c1c0
        undists0 = undists0[:, :, np.newaxis]
        temp = E @ undists0

        undists1 = undists1[:, np.newaxis, :]
        epipolar_constrains = (undists1 @ temp).flatten()
        # print(epipolar_constrains)
        print(
            f"{idx0} - {idx1} shape : {epipolar_constrains.shape[0]} /\n mean: {epipolar_constrains}"
        )
        # print(f"before epi :{kpts0.shape[0]}")
        threshold = 0.1
        if idx0 == 0 and idx1 == 2:
            threshold = 0.001
        epipolar_constrains = np.abs(epipolar_constrains) < threshold

        before_epi0 = kpts0
        kpts0 = kpts0[epipolar_constrains]
        seg_ids0 = seg_ids0[epipolar_constrains]

        before_epi1 = kpts1
        kpts1 = kpts1[epipolar_constrains]
        seg_ids1 = seg_ids1[epipolar_constrains]

        unique_ids0 = np.unique(seg_ids0.flatten())
        unique_ids1 = np.unique(seg_ids1.flatten())

        id_idx0 = {}
        for idx, id in enumerate(unique_ids0):
            id_idx0[id] = idx

        id_idx1 = {}
        for idx, id in enumerate(unique_ids1):
            id_idx1[id] = idx

        rows = unique_ids0.shape[0]
        cols = unique_ids1.shape[0]

        hit_mat = np.zeros((rows, cols))

        for id0, id1 in zip(seg_ids0, seg_ids1):
            r = id_idx0[id0]
            c = id_idx1[id1]
            hit_mat[r, c] += 1

        matched_indices = np.array(solve_dense(-hit_mat)).T
        matches = []

        for m in matched_indices:
            if hit_mat[m[0], m[1]] > 4:
                matches.append((unique_ids0[m[0]], unique_ids1[m[1]]))

        segment_id_pair += matches

        img0 = image_list[idx0]
        img1 = image_list[idx1]

        stitched_img = np.concatenate([img0, img1], axis=1)
        vis_img = stitched_img.copy()

        vis_lines_before = np.concatenate(
            [before_epi0, before_epi1], axis=1
        ).astype(np.int32)
        vis_lines_before[:, 2] += img0.shape[1]

        vis_lines = np.concatenate([kpts0, kpts1], axis=1).astype(np.int32)
        vis_lines[:, 2] += img0.shape[1]

        for i, line in enumerate(vis_lines_before):
            c = (0, 0, 0)
            line = line.tolist()
            pt1, pt2 = (line[0], line[1]), (line[2], line[3])
            cv2.line(vis_img, pt1, pt2, color=c, lineType=cv2.LINE_AA)
            cv2.circle(vis_img, pt1, radius=4, color=c, lineType=cv2.LINE_AA)
            cv2.circle(vis_img, pt2, radius=4, color=c, lineType=cv2.LINE_AA)

        for i, line in enumerate(vis_lines):
            c = colors[i % len(colors)]
            line = line.tolist()
            pt1, pt2 = (line[0], line[1]), (line[2], line[3])
            cv2.line(vis_img, pt1, pt2, color=c, lineType=cv2.LINE_AA)
            cv2.circle(vis_img, pt1, radius=4, color=c, lineType=cv2.LINE_AA)
            cv2.circle(vis_img, pt2, radius=4, color=c, lineType=cv2.LINE_AA)

        cv2.imwrite(
            f'./cache/{data_base_name}/match_{timestamp}_{idx0}_{idx1}.png',
            vis_img,
        )

        # break

    del torch_images, features, preds
    torch.cuda.empty_cache()

    segment_id_pairs_dict[timestamp] = segment_id_pair
    # print(preds)
    # preds = None
    # break
    # graph_tracker.track(preds)
    # matches = e2emvm.extract_matched_features(preds, graph_tracker)
    # vis = e2emvm.vis(image_list, matches)

    # idx = 0
    # for key, val in vis.items():
    #     cv2.imwrite(f"./cache/{data_base_name}/result{count}_{idx}_pair.png", val)
    #     idx += 1
    # cv2.waitKey(5000)

unique_edges = set()

for key, edges in segment_id_pairs_dict.items():
    for edge in edges:
        # print(edge)
        unique_edges.add(tuple(sorted(edge)))

from collections import defaultdict

graph = defaultdict(list)
for u, v in unique_edges:
    graph[u].append(v)
    graph[v].append(u)


def find_connected_components(graph):
    visited = set()
    components = []

    def dfs(node, component):
        visited.add(node)
        component.append(node)
        for neighbor in graph[node]:
            if neighbor not in visited:
                dfs(neighbor, component)

    for node in graph:
        if node not in visited:
            component = []
            dfs(node, component)
            components.append(component)

    return components


connected_components = find_connected_components(graph)


def merge_seg_tracklet(trls: List[SegmentTracklet]):
    merged_trl = tracklet_database.create_new_segment_tracklet()
    for trl in trls:
        merged_trl.cam_id_to_segment_mask.update(trl.cam_id_to_segment_mask)
        tracklet_database.delete_segment_tracklet(trl.id)
        for cam_id, mask in trl.cam_id_to_segment_mask.items():
            cam_state = cam_states_dict[cam_id.timestamp][cam_id.index]
            del cam_state.seg_trls[trl.id]
            cam_state.seg_trls[merged_trl.id] = merged_trl

    merged_trl.merged = True


for connected_component in connected_components:
    to_merge_trl = []

    for id in connected_component:
        to_merge_trl.append(tracklet_database.query_segment_tracklet(id))

    merge_seg_tracklet(to_merge_trl)

video_writer = cv2.VideoWriter(
    f"./cache/{data_base_name}/{os.path.basename(data_path)}_merged_tracklet.mp4",
    cv2.VideoWriter_fourcc(*'mp4v'),
    20,
    (IMG_W, IMG_H),
)
# for i in range(20):

for timestamp, cam_states in cam_states_dict.items():
    print(f"saving {timestamp}")

    frame = np.zeros((IMG_H, IMG_W, 3), dtype=np.uint8)

    for idx, cam_state in enumerate(cam_states):

        image = cam_state.draw_merged_tracklet()

        pos = video_layout[idx]
        frame[pos[1] : pos[1] + img_h, pos[0] : pos[0] + img_w] = image

    video_writer.write(frame)


video_writer.release()

# video_writer = cv2.VideoWriter(
#     f"./cache/{data_base_name}/{os.path.basename(data_path)}_tracklet.mp4",
#     cv2.VideoWriter_fourcc(*'mp4v'),
#     20,
#     (IMG_W, IMG_H),
# )
# # for i in range(20):

# for timestamp, cam_states in cam_states_dict.items():
#     print(f"saving {timestamp}")

#     frame = np.zeros((IMG_H, IMG_W, 3), dtype=np.uint8)

#     for idx, cam_state in enumerate(cam_states):

#         image = cam_state.draw_tracklet()

#         pos = video_layout[idx]
#         frame[pos[1] : pos[1] + img_h, pos[0] : pos[0] + img_w] = image

#     video_writer.write(frame)
# video_writer.release()

##### optical flow #####
target_idx = 2
video_writer = cv2.VideoWriter(
    f"./cache/{data_base_name}/{os.path.basename(data_path)}_flow_{target_idx}.mp4",
    cv2.VideoWriter_fourcc(*'mp4v'),
    20,
    (832, 468),
)
# for i in range(20):
prev_image = None
for timestamp, cam_states in cam_states_dict.items():
    print(f"saving {timestamp}")

    if prev_image is None:
        prev_image = cv2.cvtColor(
            cam_states[target_idx].image, cv2.COLOR_RGB2GRAY
        )
        continue

    curr_image = cv2.cvtColor(cam_states[target_idx].image, cv2.COLOR_RGB2GRAY)

    flow = cv2.calcOpticalFlowFarneback(
        prev_image,
        curr_image,
        None,
        0.5,
        4,
        30,
        3,
        5,
        1.2,
        0,
    )

    frame = visualize_optical_flow(prev_image, flow)

    prev_image = curr_image

    video_writer.write(frame)
video_writer.release()
##### optical flow done #####


# ## record tracking
# W, H = 0, 0
# fps = 20

# idx_len = 0
# for timestamp, cam_states in cam_states_dict.items():
#     H, W = cam_states[0].draw_merged_tracklet().shape[:2]
#     # H, W = cam_states[0].draw_scene().shape[:2]
#     idx_len = len(cam_states)
#     break

# video_writers = []
# for idx in range(idx_len):
#     video_writer = cv2.VideoWriter(
#         f"./cache/{data_base_name}/{os.path.basename(data_path)}_{idx}.mp4",
#         cv2.VideoWriter_fourcc(*'mp4v'),
#         fps,
#         (W, H),
#     )
#     video_writers.append(video_writer)

# for timestamp, cam_states in cam_states_dict.items():
#     print(f"saving {timestamp}")
#     for idx, cam_state in enumerate(cam_states):

#         image = cam_state.draw_merged_tracklet()
#         # image = cam_state.draw_scene()
#         video_writers[idx].write(image)


# for video_writer in video_writers:
#     video_writer.release()

processing 1130726
0 - 1 shape : 29 /
 mean: [ 7.4279e-05  0.00015804  0.00025497 -8.0805e-05 -6.1818e-05  0.00012705  3.0704e-05  0.00021009  0.00012974  4.5003e-05  0.00020336  4.4439e-06  8.0634e-05  1.8115e-05  -7.676e-05  9.0086e-05  9.3884e-05 -6.3144e-05  0.00017862  3.4165e-05  3.9646e-05  0.00011811  0.00012964  0.00038635  -6.988e-05   1.213e-05
 -7.2318e-05  6.6002e-05  2.2852e-06]
0 - 2 shape : 15 /
 mean: [ 0.00012051 -0.00010738   0.0004705 -3.8353e-05  -0.0003718 -0.00033184 -0.00025847  3.0149e-05  0.00036007  0.00012493   0.0010282  -0.0003014  1.0769e-05  0.00020733 -0.00092052]
2 - 3 shape : 31 /
 mean: [   0.078154    0.058648    0.072465    0.083938    0.043983    0.052452    0.069193    0.041931    0.050611    0.030342    0.013929     0.06305    0.026159    0.043616    0.059861    0.066797    0.031403    -0.21783    0.020464   -0.026132    0.047673    0.040093    0.053649     0.14155    0.040907    -0.78442
     -1.0833    -0.96811     -1.3167     -1.1819     -1.3

In [12]:
import itertools

list(itertools.combinations(3, 2))

TypeError: 'int' object is not iterable